# Setup

In [ ]:
%%capture
# 1. Install dependencies
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio openai
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()


In [ ]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = ''

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

In [17]:
import httpx
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent
from pydantic import BaseModel

MODEL = OpenAIChatModel(
    'anthropic/claude-haiku-4.5',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ),
)

# Tools

A **plain LLM call** has no memory of the world after its training cutoff and no tools. An **agent** has a loop: it can call tools, see the results, and reason over them.

In [18]:
# (a) Plain agent — no tools
from pydantic_ai import Agent

plain_agent = Agent(MODEL)
result = plain_agent.run_sync('What is today\'s date?')
print(result.output)

I don't have access to real-time information, so I can't tell you today's date. You can check your device's calendar, clock, or ask your phone's voice assistant for the current date.


In [19]:
# (b) Agent with one tool
from datetime import date

tool_agent = Agent(MODEL)

@tool_agent.tool_plain
def get_today() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return date.today().isoformat()

result = tool_agent.run_sync('What is today\'s date?')
print(result.output)

Today's date is **May 23, 2026** (2026-05-23).


**What just happened?**

The first agent had no way to know the current date. The second agent had a tool — `get_today()` — and the LLM decided to call it.

Critically, the LLM doesn't execute the function itself. It returns a *tool-call request*, the framework runs the function, the result is appended to the conversation, and the LLM then produces its final answer.


# Trace 

In [20]:
print(f'Final answer: {result.output}\n')
print('Trace (each ModelMessage in the conversation):')

def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)[:200]
            print(f'    └─ {kind}: {snippet}')

pretty_print_trace(result)

Final answer: Today's date is **May 23, 2026** (2026-05-23).

Trace (each ModelMessage in the conversation):

[0] ModelRequest
    └─ UserPromptPart: UserPromptPart(content="What is today's date?", timestamp=datetime.datetime(2026, 5, 23, 14, 49, 51, 11810, tzinfo=datetime.timezone.utc))

[1] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='get_today', args='{}', tool_call_id='toolu_bdrk_018z1KGTicZLRT3WXBQKBHJr')

[2] ModelRequest
    └─ ToolReturnPart: ToolReturnPart(tool_name='get_today', content='2026-05-23', tool_call_id='toolu_bdrk_018z1KGTicZLRT3WXBQKBHJr', timestamp=datetime.datetime(2026, 5, 23, 14, 49, 51, 918323, tzinfo=datetime.timezone.ut

[3] ModelResponse
    └─ TextPart: TextPart(content="Today's date is **May 23, 2026** (2026-05-23).")


# Exercise

In [21]:
# Add a tool to the tool_agent called count_words that returns the number of whitespace-separated words in a piece of text.
tool_agent = Agent(MODEL)


In [ ]:
# Test
text_to_count = ""
result = tool_agent.run_sync(f"how many words are in this text? : {text_to_count}")
pretty_print_trace(result)

In [ ]:
# BONUS: Add a tool that will count the number of the letter 'r' in a word

In [ ]:
# Test
result = tool_agent.run_sync(f"how many r's are in the word strawberry?")
pretty_print_trace(result)